## Future projections - Bezerra map

Use the projections from Bezerra et al to predict the future biomass accumulated by secondary forests.


In [1]:
import ee
import geemap
from utils import *

initialize()
config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder
last_year = config.last_year

ModuleNotFoundError: No module named 'utils'

In [ ]:
ssp1_2015 = ee.Image("projects/forestregrowth/assets/projections/forest_2015_SSP1_RCP19")
ssp1_2050 = ee.Image("projects/forestregrowth/assets/projections/forest_2050_SSP1_RCP19")

ssp2_2015 = ee.Image("projects/forestregrowth/assets/projections/forest_2015_SSP2_RCP45")
ssp2_2050 = ee.Image("projects/forestregrowth/assets/projections/forest_2050_SSP2_RCP45")

ssp3_2015 = ee.Image("projects/forestregrowth/assets/projections/forest_2015_SSP3_RCP70")
ssp3_2050 = ee.Image("projects/forestregrowth/assets/projections/forest_2050_SSP3_RCP70")

# get the area increase per 10km grid cell
ssp1_delta = ssp1_2050.subtract(ssp1_2015).rename("ssp1_delta").clip(amazon.geometry())
ssp2_delta = ssp2_2050.subtract(ssp2_2015).rename("ssp2_delta").clip(amazon.geometry())
ssp3_delta = ssp3_2050.subtract(ssp3_2015).rename("ssp3_delta").clip(amazon.geometry())

# make one prediction per 10km grid cell.
age = 35
# get average fire history per 10km grid cell

fire = (ee.Image("projects/mapbiomas-public/assets/brazil/fire/collection3/mapbiomas_fire_collection3_annual_burned_coverage_v1")
    .select([f"burned_coverage_{year}" for year in config.range_1985_2020])
    .byte()
    .rename([str(year) for year in config.range_1985_2020])
    .gt(0)
    .reduce('sum').rename("num_fires")).unmask(0)

# Aggregate the high-resolution pixels into the 10 km grid
fire_10k = fire.reduceResolution(
    reducer = ee.Reducer.mean(),
    maxPixels = 1024,
    bestEffort = True # Use all pixels that can fit in the larger pixel
).reproject(
    crs = 'EPSG:4326',
    scale = 10000
).rename("fire")

sur_cover = ee.Image(f"{data_folder}/sur_cover") # should I get the 10k average as well?

# before I was getting the area per km, making one prediction per km, and assuming they would be the same for all pixels in the 1km grid cell. Now I am doing the same, just with 10km - or should I aggregate to do something 'averaged' out for the entire 100km2?

# Aggregate the high-resolution pixels into the 10 km grid
sur_cover_10k = sur_cover.reduceResolution(
    reducer = ee.Reducer.mean(),
    maxPixels = 1024,
    bestEffort = True # Use all pixels that can fit in the larger pixel
).reproject(
    crs = 'EPSG:4326',
    scale = 10000
).rename("sur_cover")

# map = geemap.Map()
# map.addLayer(sur_cover, {'min':0, 'max':1, 'palette':['white', 'green']}, "SUR Cover")
# map.addLayer(sur_cover_10k, {'min':0, 'max':1, 'palette':['white', 'green']}, "SUR Cover 10k")
# map

mature_biomass_10k = ee.Image(f"{data_folder}/mature_biomass_10k")

# terraclim = ee.Image(f"{data_folder}/terraclim_1958_2019")

# sur_cover_10k = sur_cover.reduceResolution(
#     reducer = ee.Reducer.mean(),
#     maxPixels = 1024,
#     bestEffort = True # Use all pixels that can fit in the larger pixel
# ).reproject(
#     crs = 'EPSG:4326',
#     scale = 10000
# ).rename("sur_cover")

export_image = mature_biomass_10k.addBands(fire_10k).addBands(sur_cover_10k).addBands(ssp1_delta).addBands(ssp2_delta).addBands(ssp3_delta)

# map = geemap.Map()
# map.addLayer(export_image, {}, "Export Image")
# map.addLayer(mature_biomass_10k, {}, "Mature Biomass 10k")
# map.addLayer(ssp1_delta, {}, "SSP1 Delta")
# map
